[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/burzynski-lab/OU_PHYS2222_ComputationalPhysics/blob/main/LectureNotes/03_Libraries_and_Documentation.ipynb)

# Using Libraries and Reading Documentation

**PHYS 2222: Computational Physics** · University of Oklahoma · Spring 2027 · Week 3

> **Before you start:** in Colab, choose **File → Save a copy in Drive** so your changes are kept.

Nobody writes everything from scratch. One of Python's great strengths is the enormous collection of **libraries**: code that other people have written, tested, and documented, which you can use with a single `import` line. A large part of being a good programmer is knowing how to find the right tool and how to read its documentation.

This notebook first adds a few more of Python's built-in data types (tuples, dictionaries, and a bit more about strings). Then it covers how to import libraries, how to look up what a function does, and how to use `scipy.constants` so you never have to type the value of Planck's constant by hand again. It ends with some advice on writing clean code.

## Learning goals

By the end of this notebook you should be able to:

- use **tuples** and **dictionaries**, and a few common **string methods**;
- import modules in different ways (`import math`, `from math import sqrt`, `import numpy as np`) and explain what a namespace is;
- tell the difference between Python's standard library and third-party packages;
- find out what a function does using `help()`, `?`, docstrings, and the online documentation;
- look up physical constants, with units and uncertainties, in `scipy.constants`;
- use complex numbers and know when to use `cmath` instead of `math`;
- organize your own functions into a reusable "toolbox", and follow basic good coding practices;
- compute de Broglie wavelengths and Compton shifts.

## 1. Tuples

A **tuple** is like a list, but written with parentheses instead of square brackets, and it **cannot be changed** after it is created (it is *immutable*). Use a tuple for a small, fixed group of values that belong together, like the $(x, y, z)$ coordinates of a point, or a measured value with its uncertainty.

In [ ]:
position = (1.0, 2.5, -0.3)       # (x, y, z) in meters
measurement = (9.79, 0.04)        # g and its uncertainty, m/s^2

print(position[0])                # indexing works just like lists
print(len(position))
print(type(position))

Trying to change an element of a tuple gives an error. (We use `try`/`except` from Notebook 02 so that the notebook keeps running.)

In [ ]:
try:
    position[0] = 5.0
except TypeError as err:
    print("TypeError:", err)

### Tuple unpacking

You can assign the elements of a tuple to several variables at once. This is called **unpacking**:

In [ ]:
x, y, z = position
print(f"x = {x} m, y = {y} m, z = {z} m")

g, sigma_g = measurement
print(f"g = {g} +/- {sigma_g} m/s^2")

You have actually used tuples already. In Notebook 02, `drop_ball` ended with `return times, heights`. Writing several values separated by commas creates a tuple (the parentheses are optional), and `times, heights = drop_ball(...)` unpacked it. That is how a Python function returns more than one value:

In [ ]:
import math     # for cos and sin (Section 4 explains imports in detail)

def polar_to_cartesian(r, theta):
    """Convert polar coordinates (r, theta in radians) to Cartesian (x, y)."""
    return r * math.cos(theta), r * math.sin(theta)

x, y = polar_to_cartesian(2.0, 0.5)
print(x, y)

## 2. Dictionaries

A list finds things by **position** (index 0, 1, 2, ...). Often you would rather find things by **name**. A **dictionary** (type `dict`) stores **key-value pairs**: you look up a value by its key, like looking up a word in a real dictionary to find its definition.

Dictionaries are written with curly braces `{}`, with a colon between each key and its value:

In [ ]:
# Rest energies of some particles, in MeV (rounded)
mass_MeV = {
    "electron": 0.511,
    "muon": 105.7,
    "proton": 938.3,
}

print(mass_MeV["muon"])       # look up a value by its key
print(len(mass_MeV))          # number of key-value pairs

The keys are usually strings (but numbers and tuples work too). Each key appears only once. You can add new pairs or change existing ones by assignment:

In [ ]:
mass_MeV["neutron"] = 939.6      # add a new key
mass_MeV["electron"] = 0.51100   # replace an existing value
print(mass_MeV)

Asking for a key that is not there gives a `KeyError`. Use the `in` operator to check first, or the `.get` method, which returns a default value instead of crashing:

In [ ]:
print("proton" in mass_MeV)
print("photon" in mass_MeV)
print(mass_MeV.get("photon", "not in the dictionary"))

try:
    print(mass_MeV["photon"])
except KeyError as err:
    print("KeyError:", err)

### Looping over a dictionary

`.keys()` gives the keys, `.values()` the values, and `.items()` gives `(key, value)` pairs as tuples, which you can unpack right in the `for` statement. Looping directly over a dictionary gives its keys.

In [ ]:
for name, mass in mass_MeV.items():
    print(f"{name:>10}: {mass:8.3f} MeV")

**Lists vs. tuples vs. dictionaries:**

| Type | Written as | Find things by | Can change? | Use for |
|---|---|---|---|---|
| list | `[1, 2, 3]` | position | yes | sequences of data, results you build up in a loop |
| tuple | `(1, 2, 3)` | position | no | small fixed groups: coordinates, (value, error) |
| dict | `{"a": 1}` | key | yes | named quantities: parameters, lookup tables |

## 3. A few string methods

Strings have many useful **methods** (functions called with a dot after the string). Here are a few you will meet when handling labels, file names, and data:

In [ ]:
label = "  Hydrogen Balmer series  "

print(label.strip())                    # remove spaces at both ends
print(label.strip().lower())            # lowercase
print(label.strip().upper())            # uppercase
print(label.replace("Balmer", "Lyman")) # replace text
print("Balmer" in label)                # test whether one string contains another

In [ ]:
line = "656.5, 486.3, 434.2, 410.3"     # a line of comma-separated numbers, as text
parts = line.split(",")                 # split into a list of strings at each comma
print(parts)

wavelengths = [float(p) for p in parts] # convert each piece to a number
print(wavelengths)

print(" | ".join(["e", "mu", "tau"]))  # join a list of strings with a separator

Strings can also be indexed and sliced like lists: `"electron"[0]` is `"e"` and `"electron"[-3:]` is `"ron"`. Like tuples, strings are immutable: methods such as `.upper()` return a *new* string rather than changing the original.

### Your turn 1

Make a dictionary `half_life_s` with the half-lives, in seconds, of a few unstable particles or nuclei: the free neutron (about 611 s), the muon ($1.523\times10^{-6}$ s, which is $\ln 2$ times its 2.197 μs mean lifetime), and carbon-14 (5730 years; convert to seconds). Then loop over it and print each name with its half-life in scientific notation, and also the fraction left after one hour, $(1/2)^{t/T_{1/2}}$.

In [ ]:
# Your code here

## 4. Modules and import

A **module** is a file of Python code (functions, constants, ...) that you can load into your program. A **package** is a collection of related modules. You load them with `import`. There are several styles, and you will see all of them in other people's code.

### Style 1: `import module`

In [ ]:
import math

print(math.sqrt(16))
print(math.pi)

Everything in the module is accessed with the module name and a dot: `math.sqrt`, `math.pi`. The module acts as a **namespace**: a labeled container for names. This is a good thing: you can always see where a function came from, and two modules can each have a function called `sqrt` without confusion.

You can list everything a module contains with `dir`:

In [ ]:
print(dir(math))

(Ignore the names that start and end with double underscores; those are for Python's internal use.)

### Style 2: `from module import name`

This imports specific names directly, so you can use them without the module prefix:

In [ ]:
from math import sqrt, pi, cos

print(sqrt(2))
print(cos(pi))

This is shorter, but it is less clear where `sqrt` came from, and it can cause **name clashes**: if two modules both have a `sqrt` (and `math`, `cmath`, and `numpy` all do!), the one imported last silently wins.

You may also see `from math import *`, which imports *every* name from the module. **Avoid this.** It fills your notebook with hundreds of names you cannot see, and makes clashes very likely.

### Style 3: `import module as alias`

This imports a module under a shorter name. Some aliases are so universal that everyone uses them, and you should too:

In [ ]:
import numpy as np                 # the standard alias for NumPy
import matplotlib.pyplot as plt    # the standard alias for Matplotlib's plotting module

print(np.sqrt(2))
print(np.pi)

NumPy (Numerical Python) is the foundation of almost all scientific computing in Python, and it is the subject of the next notebook. Matplotlib is for plotting (Notebook 05). Note that `matplotlib.pyplot` shows that packages can contain modules inside them, separated by dots.

**Where to put imports:** by convention, put all your imports in one cell near the top of a notebook. Then anyone reading it (including your grader) can see at a glance what it depends on, and "Run all" always loads them first.

### The standard library vs. third-party packages

- The **standard library** comes with every Python installation. Examples: `math` (math functions), `cmath` (complex math), `random` (random numbers), `statistics`, `os` and `pathlib` (files and folders), `time` and `datetime`, `csv`, `json`. See the [list of standard library modules](https://docs.python.org/3/library/index.html).
- **Third-party packages** are written by others and must be installed separately. The scientific Python "stack" includes `numpy`, `scipy` (scientific algorithms: integration, optimization, special functions, constants, ...), `matplotlib`, `pandas` (tables of data), `scikit-learn` (machine learning), and `torch` (neural networks).

On your own computer you install third-party packages with a tool called `pip` (or `uv`). In Colab, all the packages used in this course are **already installed**, so you never need to install anything. (If you ever do, a code cell containing `!pip install packagename` runs the installer; the `!` sends the line to the computer's command line instead of to Python.)

If you import a package that is not installed, or misspell its name, you get:

```
ModuleNotFoundError: No module named 'nunpy'
```

## 5. Reading documentation

No one memorizes every function and its arguments. Programmers look things up constantly, and you should too. Here are four ways, from quickest to most thorough.

### (a) `help()`

`help(something)` prints the documentation for a function or module, which comes mostly from its **docstring**:

In [ ]:
help(math.sqrt)

In [ ]:
help(round)

How to read this:

- The first line is the **signature**: the function's name and arguments. `round(number, ndigits=None)` means `round` needs one argument, `number`, and has an optional second one, `ndigits`, whose default value is `None`.
- (A `/` in a signature means the arguments before it must be given by position, not by name. You can ignore it for now.)
- Below is a description of what the function does and returns.

So now we know `round` can round to a given number of decimal places:

In [ ]:
print(round(2.718281828, 3))

`help` works on **your own functions too**, and shows the docstring you wrote. That is a big reason to write docstrings.

In [ ]:
def lorentz_gamma(beta):
    """Return the Lorentz factor gamma = 1/sqrt(1 - beta^2) for a speed beta = v/c.

    beta must satisfy -1 < beta < 1.
    """
    return 1 / math.sqrt(1 - beta**2)

help(lorentz_gamma)

### (b) The `?` shortcut in Jupyter and Colab

In a notebook, putting a `?` after a name and running the cell shows the same information in a pop-up or side panel (in Colab) instead of in the cell output. Two question marks, `??`, show the source code too, if it is available. Run the cell below and look for the help panel.

In [ ]:
math.hypot?

### (c) Tab completion and tooltips

In Colab, if you type the start of a name and press **Tab** (or just pause), a list of possible completions pops up. Type `math.` and pause to see everything in the `math` module. When you type the opening parenthesis of a function call, Colab also shows a tooltip with the function's signature and docstring. This is plain Python introspection, not the AI completion you turned off in Notebook 00; it is fine to use.

### (d) The online documentation

For anything beyond a quick reminder, use the official documentation, which has longer explanations and examples:

- Python and its standard library: [docs.python.org/3](https://docs.python.org/3/)
- NumPy: [numpy.org/doc](https://numpy.org/doc/stable/)
- SciPy: [docs.scipy.org](https://docs.scipy.org/doc/scipy/)
- Matplotlib: [matplotlib.org](https://matplotlib.org/stable/)

A search such as "numpy linspace" or "python round function" usually takes you straight to the right page. A function's documentation page follows a standard pattern: the **signature**, a one-line summary, the **Parameters** (each argument, its type, and its meaning), what it **Returns**, sometimes **Notes** on the method used, and **Examples**. The examples are often the fastest way to understand a function; try copying one into a cell.

### Your turn 2

Use `help()` (or `?`) to find out what the `math` functions `hypot`, `factorial`, and `log` do. In particular, find out how to use `math.log` to compute a logarithm in base 2. Then compute: the length of the diagonal of a 3 m by 4 m by 12 m box (check whether `hypot` accepts three arguments), $10!$, and $\log_2 1024$.

In [ ]:
# Your code here

## 6. Physical constants with `scipy.constants`

So far we have typed constants like $h$ and $c$ by hand. That is slow and risky (one typo in an exponent and your answer is off by a factor of 10). The `scipy.constants` module contains the internationally recommended values of the fundamental constants, the **CODATA** values, in SI units.

In [ ]:
from scipy import constants

print(constants.c)        # speed of light, m/s
print(constants.h)        # Planck constant, J s
print(constants.hbar)     # reduced Planck constant h/(2 pi), J s
print(constants.e)        # elementary charge, C
print(constants.m_e)      # electron mass, kg
print(constants.m_p)      # proton mass, kg
print(constants.k)        # Boltzmann constant, J/K
print(constants.G)        # Newton's gravitational constant, m^3/(kg s^2)
print(constants.epsilon_0)   # vacuum permittivity, F/m

Some people prefer to import the constants they need directly, which makes formulas look more like the math:

In [ ]:
from scipy.constants import c, h, hbar, e, m_e, m_p, eV

print(f"electron rest energy: {m_e * c**2 / eV / 1e6:.6f} MeV")

Here `eV` is the value of one electron-volt in joules, so dividing by `eV` converts J to eV. `scipy.constants` also contains **unit prefixes and conversion factors**, which make code read naturally: `constants.nano` is `1e-9`, `constants.pico` is `1e-12`, `constants.mega` is `1e6`, `constants.angstrom` is `1e-10`, and `constants.year` is the number of seconds in a year.

In [ ]:
lam = 500 * constants.nano          # 500 nm, in m
E_photon = h * c / lam / eV         # in eV
print(f"A 500 nm photon has E = {E_photon:.3f} eV")
print(f"One year is {constants.year:.4e} s")

### The `physical_constants` dictionary

The short names above cover the most common constants. The full CODATA list, several hundred constants, lives in a **dictionary** called `constants.physical_constants`. Each key is the constant's full name (a string), and each value is a **tuple** of `(value, unit, uncertainty)`. Now you can see why we learned dictionaries and tuples first!

In [ ]:
pc = constants.physical_constants      # a short name for this long dictionary

print(pc["electron mass"])
print(pc["Rydberg constant"])
print(pc["electron mass energy equivalent in MeV"])

In [ ]:
value, unit, uncertainty = pc["proton mass energy equivalent in MeV"]
print(f"m_p c^2 = {value} +/- {uncertainty} {unit}")

How do you find the exact key name? Use `constants.find`, which searches the keys for a piece of text and returns a list of matching names:

In [ ]:
print(constants.find("Rydberg"))
print(constants.find("Bohr"))

The Rydberg energy we used in Notebook 02 is the key `"Rydberg constant times hc in eV"`, and the Bohr radius is `"Bohr radius"`:

In [ ]:
E_R = pc["Rydberg constant times hc in eV"][0]     # [0] picks the value out of the tuple
a_0 = pc["Bohr radius"][0]
print(f"Rydberg energy: {E_R:.6f} eV")
print(f"Bohr radius:    {a_0:.6e} m")

**About versions:** CODATA updates its recommended values every four years, and newer versions of SciPy use newer values. So the last few digits of some constants (like the electron mass) may differ slightly between your Colab session and someone else's computer. Constants such as $c$, $h$, $e$, and $k$ are *exact* by definition of the SI units and never change.

**From now on in this course, use `scipy.constants` rather than typing constants by hand.**

### A dictionary of particle masses

Let us build our own dictionary of rest energies, in MeV, pulling the values from CODATA where possible. Pion masses are not in CODATA, so we take them from the Particle Data Group (PDG) *Review of Particle Physics*, and we say so in a comment.

In [ ]:
particle_mass_MeV = {
    "electron": pc["electron mass energy equivalent in MeV"][0],
    "muon":     pc["muon mass energy equivalent in MeV"][0],
    "tau":      pc["tau mass energy equivalent in MeV"][0],
    "proton":   pc["proton mass energy equivalent in MeV"][0],
    "neutron":  pc["neutron mass energy equivalent in MeV"][0],
    "alpha":    pc["alpha particle mass energy equivalent in MeV"][0],
    "pi+":      139.57,    # charged pion, PDG
    "pi0":      134.98,    # neutral pion, PDG
}

for name, m in particle_mass_MeV.items():
    ratio = m / particle_mass_MeV["electron"]
    print(f"{name:>9}: {m:10.3f} MeV   ({ratio:8.1f} electron masses)")

## 7. `math` vs. `cmath`: complex numbers

**Complex numbers** are essential in quantum mechanics: wavefunctions like $\psi(x) = e^{ikx}$ are complex. Python has complex numbers built in. It writes the imaginary unit $i$ as `j` (the engineering convention), attached directly to a number: `1j`, `3 + 4j`.

In [ ]:
z = 3 + 4j
print(z, type(z))
print(z.real, z.imag)     # real and imaginary parts
print(abs(z))             # magnitude |z| = sqrt(3^2 + 4^2)
print(z.conjugate())      # complex conjugate z*
print(z * z.conjugate())  # |z|^2, as a complex number with zero imaginary part

The functions in `math` only work with real numbers. For example, `math.sqrt(-1)` gives a `ValueError` (try it inside `try`/`except` below). The **`cmath`** module ("complex math") has versions of the same functions that accept and return complex numbers.

In [ ]:
import cmath

try:
    print(math.sqrt(-1))
except ValueError as err:
    print("math.sqrt(-1) failed:", err)

print("cmath.sqrt(-1) =", cmath.sqrt(-1))

Euler's formula, $e^{i\theta} = \cos\theta + i\sin\theta$, is everywhere in physics. With $\theta = \pi$ it gives Euler's identity $e^{i\pi} = -1$:

In [ ]:
print(cmath.exp(1j * math.pi))     # -1, plus round-off in the imaginary part
print(cmath.exp(1j * math.pi / 2)) # i
print(abs(cmath.exp(1j * 0.73)))   # |e^{i theta}| = 1 for any real theta

Notice the imaginary part `1.2246467991473532e-16` instead of exactly zero: round-off once more. The rule of thumb is to use `math` when your numbers are real (it is faster, and an error is a useful warning that something has gone unphysical, like a speed faster than light) and `cmath` when you really want complex results. NumPy, which we start next week, handles complex numbers too.

## 8. Physics with constants: de Broglie wavelengths

**Physics recap.** In 1924 de Broglie proposed that *every* particle has a wavelength, related to its momentum $p$ by
$$\lambda = \frac{h}{p}.$$
For a slow particle, $p = mv$. It is often easier to know the kinetic energy $K$ than the speed: for example, an electron accelerated through a voltage $V$ gains $K = eV$. Nonrelativistically $K = p^2/(2m)$, so $p = \sqrt{2mK}$. The wavelength matters because waves *diffract* when they pass through structures about the size of their wavelength. In 1927 Davisson and Germer confirmed de Broglie's idea by diffracting 54 eV electrons from a nickel crystal, whose atoms are spaced about 0.2 nm apart.

In [ ]:
def de_broglie_wavelength(mass, speed):
    """Return the (nonrelativistic) de Broglie wavelength in m
    for a particle of mass (kg) moving at speed (m/s)."""
    return constants.h / (mass * speed)

def de_broglie_from_K(mass, K_eV):
    """Return the nonrelativistic de Broglie wavelength in m
    for a particle of mass (kg) with kinetic energy K_eV (eV)."""
    K = K_eV * constants.eV               # convert eV to J
    p = math.sqrt(2 * mass * K)
    return constants.h / p

In [ ]:
lam_e = de_broglie_from_K(constants.m_e, 54)
print(f"54 eV electron:   lambda = {lam_e / constants.nano:.4f} nm")

lam_ball = de_broglie_wavelength(0.145, 40)
print(f"baseball, 40 m/s: lambda = {lam_ball:.2e} m")

The electron's wavelength is comparable to the spacing of atoms in a crystal, so it diffracts. The baseball's wavelength is about $10^{-34}$ m, far smaller than an atomic nucleus ($\sim10^{-15}$ m), which is why you never see baseballs diffract.

### Relativistic momentum

For fast particles we need relativity. From Notebook 01, $E = K + mc^2$ and $E^2 = (pc)^2 + (mc^2)^2$. Solving for $p$:
$$pc = \sqrt{E^2 - (mc^2)^2} = \sqrt{K^2 + 2K\,mc^2}.$$
For $K \ll mc^2$ this reduces to the nonrelativistic $p = \sqrt{2mK}$ (check this yourself!). Let us write a function that works at any energy, using energies in MeV to keep the numbers friendly.

In [ ]:
def de_broglie_relativistic(mc2_MeV, K_MeV):
    """Return the relativistic de Broglie wavelength in m for a particle with
    rest energy mc2_MeV (MeV) and kinetic energy K_MeV (MeV)."""
    pc_MeV = math.sqrt(K_MeV**2 + 2 * K_MeV * mc2_MeV)    # pc in MeV
    pc_J = pc_MeV * constants.mega * constants.eV          # pc in J
    return constants.h * constants.c / pc_J               # lambda = h/p = hc/(pc)

In [ ]:
# Electrons in a transmission electron microscope have K of about 0.1 to 0.3 MeV
mc2 = particle_mass_MeV["electron"]
K = 0.2
lam_rel = de_broglie_relativistic(mc2, K)
lam_nonrel = de_broglie_from_K(constants.m_e, K * 1e6)

print(f"200 keV electron: relativistic     {lam_rel / constants.pico:.3f} pm")
print(f"                  nonrelativistic  {lam_nonrel / constants.pico:.3f} pm")

At 200 keV the nonrelativistic formula is already off by about 9%. Electron microscopes are designed with relativity built in.

Now loop over our particle dictionary and compute the wavelength of each particle at the same kinetic energy of 1 MeV:

In [ ]:
K = 1.0   # MeV
print(f"de Broglie wavelengths at K = {K} MeV")
for name, mc2 in particle_mass_MeV.items():
    lam = de_broglie_relativistic(mc2, K)
    print(f"{name:>9}: {lam / constants.femto:10.2f} fm")

(1 fm $= 10^{-15}$ m, about the size of a proton.) At the same kinetic energy, heavier particles have larger momentum and so shorter wavelengths. A 1 MeV electron has a wavelength of about 870 fm, much larger than a nucleus, while a 1 MeV proton or neutron has a wavelength of about 29 fm, closer to nuclear sizes.

### Your turn 3

**Thermal neutrons** are neutrons slowed down to room temperature, with kinetic energy about $K = \frac{3}{2}k_B T$ where $k_B$ is Boltzmann's constant (`constants.k`) and $T = 300$ K. Compute their kinetic energy in eV and their de Broglie wavelength in nm, using `scipy.constants` for every constant (the neutron mass is `constants.m_n`). Compare with typical spacings between atoms in a crystal (0.1 to 0.5 nm). This is why thermal neutrons are used to study the structure of materials.

In [ ]:
# Your code here

## 9. The Compton shift

**Physics recap.** In 1923 Arthur Compton scattered X-rays off electrons and found that the scattered X-rays had a *longer* wavelength than the incoming ones. Treating light as particles (photons) colliding elastically with electrons at rest, energy and momentum conservation give the **Compton formula**
$$\Delta\lambda = \lambda' - \lambda = \frac{h}{m_e c}\left(1 - \cos\theta\right),$$
where $\theta$ is the angle through which the photon is scattered. The combination $\lambda_C = h/(m_e c)$ is the **Compton wavelength** of the electron. The shift is independent of the original wavelength, so it is only noticeable for short-wavelength X-rays and gamma rays.

In [ ]:
lambda_C = constants.h / (constants.m_e * constants.c)
print(f"Compton wavelength (computed): {lambda_C:.6e} m")
print(f"Compton wavelength (CODATA):   {pc['Compton wavelength'][0]:.6e} m")

The two agree, which is a good check that we typed the formula correctly. (Inside an f-string that uses double quotes, use single quotes for the dictionary key: `pc['Compton wavelength']`.)

In [ ]:
def compton_shift(theta_deg, mass=constants.m_e):
    """Return the Compton wavelength shift (m) for a photon scattered by
    theta_deg degrees off a particle of the given mass (kg; default: electron)."""
    theta = math.radians(theta_deg)
    return constants.h / (mass * constants.c) * (1 - math.cos(theta))

lam_in = 71.0 * constants.pico          # molybdenum K-alpha X-rays, about 71 pm (Compton's source)

print(f"{'angle (deg)':>12} {'shift (pm)':>11} {'lambda_out (pm)':>16}")
for angle in [0, 45, 90, 135, 180]:
    shift = compton_shift(angle)
    print(f"{angle:12d} {shift / constants.pico:11.3f} {(lam_in + shift) / constants.pico:16.3f}")

The maximum shift, at $\theta = 180°$ (backscattering), is $2\lambda_C \approx 4.85$ pm, which is a few percent of the X-ray wavelength and easily measurable.

### Your turn 4

(a) Use `compton_shift` with its `mass` argument to compute the maximum Compton shift for a photon scattering off a **proton** instead of an electron. By what factor is it smaller, and why?
(b) For visible light ($\lambda = 500$ nm) scattered at 90° off an electron, what is the *fractional* change $\Delta\lambda/\lambda$? Use this to explain why Compton scattering was discovered with X-rays and not visible light.

In [ ]:
# Your code here

## 10. Building your own toolbox

As the semester goes on, you will write functions that you want to reuse. A good habit is to collect related functions, with clear names and docstrings, in **one cell near the top** of your notebook, right after the imports. That cell then works like your own small module.

In [ ]:
# ---- Relativity and quantum toolbox ----
import math
from scipy import constants

MEV = constants.mega * constants.eV     # one MeV in joules

def gamma_from_beta(beta):
    """Lorentz factor for speed beta = v/c (|beta| < 1)."""
    return 1 / math.sqrt(1 - beta**2)

def beta_from_gamma(gamma):
    """Speed beta = v/c for a given Lorentz factor gamma >= 1."""
    return math.sqrt(1 - 1 / gamma**2)

def photon_energy_eV(wavelength_m):
    """Energy in eV of a photon with wavelength in m."""
    return constants.h * constants.c / wavelength_m / constants.eV

def momentum_MeV(mc2_MeV, K_MeV):
    """Momentum pc (MeV) of a particle with rest energy mc2 and kinetic energy K (MeV)."""
    return math.sqrt(K_MeV**2 + 2 * K_MeV * mc2_MeV)

In [ ]:
# Using the toolbox: a few quick checks
print(gamma_from_beta(0.6))                       # should be 1.25 exactly
print(beta_from_gamma(gamma_from_beta(0.6)))      # should give back 0.6
print(photon_energy_eV(656.5 * constants.nano))   # H-alpha, about 1.89 eV
print(momentum_MeV(0.511, 1.0))                   # 1 MeV electron, in MeV/c

Notice the checks in the second cell. Testing each function on a case where you know the answer (like $\beta = 0.6 \Rightarrow \gamma = 1.25$, or that converting there and back returns the original value) catches bugs early. Writing `MEV` in capital letters is a common convention for a **constant** that should never change.

**Going further: real modules.** If you save functions in a plain text file ending in `.py`, say `toolbox.py`, then `import toolbox` loads it as a module and you can call `toolbox.gamma_from_beta(0.6)`, exactly like `math.sqrt`. In Colab, a cell that begins with the line `%%writefile toolbox.py` saves the rest of the cell to that file. (Colab's files disappear when the runtime disconnects, so for this course it is simpler to keep your toolbox in a notebook cell.)

## 11. Good coding practices

Code is read far more often than it is written: by your grader, by your lab partner, and by you in three weeks when you have forgotten everything. Clear code is also much easier to debug. Compare these two versions of the same calculation:

In [ ]:
# Hard to read
a = 6.62607015e-34 * 299792458 / 656.5e-9 / 1.602176634e-19
b = 6.62607015e-34 * 299792458 / 486.3e-9 / 1.602176634e-19
c2 = 6.62607015e-34 * 299792458 / 434.2e-9 / 1.602176634e-19
print(a, b, c2)

In [ ]:
# Easier to read: named constants, one function, a loop, labeled output
balmer_nm = {"H-alpha": 656.5, "H-beta": 486.3, "H-gamma": 434.2}

for line_name, lam_nm in balmer_nm.items():
    E = photon_energy_eV(lam_nm * constants.nano)
    print(f"{line_name:>8}: {lam_nm} nm -> {E:.3f} eV")

The guidelines that make the difference:

1. **Meaningful names.** `photon_energy_eV` and `lam_nm` say what they are *and their units*. `a`, `b`, `c2` say nothing (and `c` would have clashed with the speed of light).
2. **Don't repeat yourself (DRY).** If you copy and paste a line and change one number, that is a sign you want a loop or a function. In the first version, a typo in one of the three copies of $h$ would go unnoticed.
3. **No "magic numbers".** Use named constants from `scipy.constants` (or a well-named variable with a comment giving the source) instead of raw numbers like `1.602176634e-19` in the middle of a formula.
4. **Small functions that do one thing**, each with a docstring. They are easy to test on their own.
5. **Comments explain *why*, not *what*.** `x += dt  # add dt to x` is useless; `# use the old velocity, so this is Euler's method` is helpful. Always comment on **units**.
6. **Test against known answers** (limits, special cases, published values).
7. **Keep the notebook runnable top to bottom:** imports and toolbox at the top, and check with **Restart session and run all**.

Following these practices is part of how your projects are graded (code clarity and organization).

## 12. Optional: the `random` module

Many simulations need random numbers: to model radioactive decays, noisy measurements, or particles entering a detector at random angles. The standard library's `random` module provides them. (From Notebook 12 on we will use NumPy's more powerful random number generator, but the ideas are the same.)

Computer "random" numbers are really **pseudo-random**: they come from a deterministic formula that produces numbers that look random. Setting the **seed** makes the sequence repeatable, which is important for debugging and for others to reproduce your results.

In [ ]:
import random

random.seed(2222)                   # fix the seed so the results are reproducible

print(random.random())              # a float uniformly distributed in [0, 1)
print(random.uniform(-1.0, 1.0))    # a float uniformly distributed in [-1, 1)
print(random.randint(1, 6))         # an integer from 1 to 6 (both included): a die roll
print(random.choice(["e", "mu", "tau"]))   # a random element of a list

**A tiny simulation.** A radioactive nucleus with half-life $T_{1/2}$ has a probability $1/2$ of decaying during any one half-life. Start with 1000 nuclei, and for each half-life, let each surviving nucleus "flip a coin" to decide whether it decays. The number left should be roughly halved each time, but with random fluctuations, just like real decay data.

In [ ]:
random.seed(1)
N = 1000                       # number of undecayed nuclei

for step in range(1, 6):
    survivors = 0
    for nucleus in range(N):
        if random.random() >= 0.5:     # probability 1/2 to survive this half-life
            survivors += 1
    N = survivors
    print(f"after {step} half-lives: {N:4d} left (expected {1000 / 2**step:6.1f})")

The simulated counts scatter around the expected values. Change the seed and rerun to see different random outcomes. You will explore this kind of **Monte Carlo** simulation in depth in Unit 3.

### Your turn 5

In the style of the toolbox in Section 10, write a function `kinetic_energy_MeV(mc2_MeV, beta)` in the cell below, returning $K = (\gamma - 1)\,mc^2$ and using `gamma_from_beta`. Give it a docstring and check it with `help`. Then use it with the `particle_mass_MeV` dictionary to print a table of the kinetic energy each particle has at $\beta = 0.5$. Finally, test it: for small $\beta$ (say 0.001) it should agree with $\frac{1}{2}\beta^2 mc^2$.

In [ ]:
# Your code here

## Summary

- **Tuples** `(a, b)`: like lists but immutable; unpack with `x, y = t`; `return a, b` returns a tuple.
- **Dictionaries** `{"key": value}`: look up by key `d["key"]`; add with `d["new"] = v`; check with `"key" in d`; `.get(key, default)`; loop with `for k, v in d.items():`.
- **String methods**: `.strip()`, `.lower()`, `.upper()`, `.replace(a, b)`, `.split(",")`, `", ".join(list)`; `"sub" in s`.
- **Imports**: `import math` (use `math.sqrt`); `from math import sqrt`; `import numpy as np`. Avoid `from module import *`. Put imports at the top.
- **Standard library** (`math`, `cmath`, `random`, `os`, ...) comes with Python; **third-party packages** (`numpy`, `scipy`, `matplotlib`, `pandas`, ...) are preinstalled in Colab.
- **Documentation**: `help(f)`, `f?` in Colab, Tab completion, and the online docs (signature, Parameters, Returns, Examples). Write docstrings so `help` works on your own functions.
- **`scipy.constants`**: `c`, `h`, `hbar`, `e`, `m_e`, `m_p`, `m_n`, `k`, `G`, `eV`; prefixes `nano`, `pico`, `femto`, `mega`; `physical_constants[name]` gives `(value, unit, uncertainty)`; `constants.find("text")` searches names.
- **Complex numbers**: `3 + 4j`, `.real`, `.imag`, `abs(z)`, `.conjugate()`; use `cmath` (e.g. `cmath.sqrt`, `cmath.exp`) for complex math, `math` for real.
- **Good practice**: meaningful names with units, no magic numbers, don't repeat yourself, small documented functions, comments that explain why, test against known answers, keep notebooks runnable top to bottom.
- `random`: `random.seed`, `random.random()`, `random.uniform(a, b)`, `random.randint(a, b)`, `random.choice(list)`.
- Physics: $\lambda = h/p$; $pc = \sqrt{K^2 + 2K\,mc^2}$; $\Delta\lambda = \frac{h}{m_e c}(1 - \cos\theta)$, $\lambda_C \approx 2.43$ pm.

## Further reading

- The official [Python tutorial](https://docs.python.org/3/tutorial/), Section 5 (data structures: tuples and dictionaries) and Section 6 (modules).
- [`scipy.constants` documentation](https://docs.scipy.org/doc/scipy/reference/constants.html), with the full list of constants and units.
- Jake VanderPlas, *A Whirlwind Tour of Python* (free online), chapters on built-in data structures and modules.
- Mark Newman, *Computational Physics*, Section 2.2.5 on modules, and Section 2.7 on good programming style.